In [5]:
import pandas as pd

data = pd.read_feather("/mnt/tonglian_data2/support_data/citic_3.fea")

In [6]:
data

TICKER_SYMBOL,INTO_DATE,000001,000002,000004,000005,000006,000007,000008,000009,000010,...,873665,873679,873690,873693,873703,873706,873726,873806,873833,874774
0,2010-01-04,全国性股份制银行,住宅物业开发,化学制剂,非住宅物业开发和运营,住宅物业开发,酒店,贸易,住宅物业开发,其他专用机械,...,None,None,None,None,None,None,None,None,None,None
1,2010-01-05,全国性股份制银行,住宅物业开发,化学制剂,非住宅物业开发和运营,住宅物业开发,酒店,贸易,住宅物业开发,其他专用机械,...,None,None,None,None,None,None,None,None,None,None
2,2010-01-06,全国性股份制银行,住宅物业开发,化学制剂,非住宅物业开发和运营,住宅物业开发,酒店,贸易,住宅物业开发,其他专用机械,...,None,None,None,None,None,None,None,None,None,None
3,2010-01-07,全国性股份制银行,住宅物业开发,化学制剂,非住宅物业开发和运营,住宅物业开发,酒店,贸易,住宅物业开发,其他专用机械,...,None,None,None,None,None,None,None,None,None,None
4,2010-01-08,全国性股份制银行,住宅物业开发,化学制剂,非住宅物业开发和运营,住宅物业开发,酒店,贸易,住宅物业开发,其他专用机械,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3789,2025-08-11,全国性股份制银行,住宅物业开发,行业应用软件,环保及水务,住宅物业开发,非住宅物业开发和运营,铁路交通设备,综合,园林工程,...,橡胶制品,其他专用机械,汽车零部件,基础件,其他专用机械,工程机械,其他专用机械,铁路交通设备,基础件,其他专用机械
3790,2025-08-12,全国性股份制银行,住宅物业开发,行业应用软件,环保及水务,住宅物业开发,非住宅物业开发和运营,铁路交通设备,综合,园林工程,...,橡胶制品,其他专用机械,汽车零部件,基础件,其他专用机械,工程机械,其他专用机械,铁路交通设备,基础件,其他专用机械
3791,2025-08-13,全国性股份制银行,住宅物业开发,行业应用软件,环保及水务,住宅物业开发,非住宅物业开发和运营,铁路交通设备,综合,园林工程,...,橡胶制品,其他专用机械,汽车零部件,基础件,其他专用机械,工程机械,其他专用机械,铁路交通设备,基础件,其他专用机械
3792,2025-08-14,全国性股份制银行,住宅物业开发,行业应用软件,环保及水务,住宅物业开发,非住宅物业开发和运营,铁路交通设备,综合,园林工程,...,橡胶制品,其他专用机械,汽车零部件,基础件,其他专用机械,工程机械,其他专用机械,铁路交通设备,基础件,其他专用机械


In [7]:
# 去掉非股票列
stock_cols = [c for c in data.columns if c not in ["TICKER_SYMBOL", "INTO_DATE"]]

# 提取股票 -> 第一个行业
industry_map = {}
for col in stock_cols:
    first_val = data[col].dropna().iloc[0] if data[col].notna().any() else None
    industry_map[col] = first_val

# 转换成 DataFrame 方便查看
industry_df = pd.DataFrame(list(industry_map.items()), columns=["Code", "Industry"])

print(industry_df.head())

     Code    Industry
0  000001    全国性股份制银行
1  000002      住宅物业开发
2  000004        化学制剂
3  000005  非住宅物业开发和运营
4  000006      住宅物业开发


In [8]:

# 统计各个行业出现的次数
industry_counts = industry_df["Industry"].value_counts()
len(industry_counts)

273

In [ ]:
import pickle
# 读取 pkl 文件
with open("/home/user96/GCN_v2_0.2/output/2023q1_subgraph_output.pkl", "rb") as f:
    graph_banchmark = pickle.load(f)

edge_banchmark = graph_banchmark["edge_index"]  # 图边 (Tensor)
node_banchmark = graph_banchmark["node_ids"]      # 股票 ID 列表

In [ ]:
import pandas as pd
import torch
from itertools import combinations

def build_graph_from_industry(
    industry_df: pd.DataFrame,
    code_col: str = "Code",
    industry_col: str = "Industry",
    keep_order: bool = False,
    undirected: bool = True,
    add_self_loops: bool = True
):
    """
    输入:
        industry_df: 至少包含 [code_col, industry_col]
        keep_order:  True=按首次出现顺序作为节点顺序; False=按代码排序
        undirected:  True=每条边添加反向(无向图用法, PyG常见)
        add_self_loops: 是否给每个节点加自环
    
    输出:
        edge_index: torch.LongTensor, shape (2, E)
        nodes: List[str], 节点id -> Code 的映射
        edge_code_pairs: List[Tuple[str, str]], 跟 edge_index 列一一对应的 (Code_u, Code_v)
    """
    df = industry_df[[code_col, industry_col]].dropna().copy()

    # 1) 生成节点列表（节点 id -> Code 的映射）
    if keep_order:
        # 按首次出现顺序去重
        nodes = df[code_col].drop_duplicates(keep="first").tolist()
    else:
        nodes = sorted(df[code_col].unique().tolist())

    code2id = {c: i for i, c in enumerate(nodes)}

    # 2) 行业内两两配对生成边
    src, dst = [], []

    for _, g in df.groupby(industry_col):
        codes = g[code_col].drop_duplicates().tolist()
        if len(codes) <= 1:
            continue
        # 两两组合
        for a, b in combinations(codes, 2):
            u, v = code2id[a], code2id[b]
            # 先加一个方向
            src.append(u); dst.append(v)
            # 若需要无向(双向边)，再加反向
            if undirected:
                src.append(v); dst.append(u)

    # 3) 可选自环
    if add_self_loops:
        for i, c in enumerate(nodes):
            src.append(i); dst.append(i)

    if len(src) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor([src, dst], dtype=torch.long)

    return edge_index, nodes


In [ ]:
edge_index, nodes = build_graph_from_industry(industry_df)


In [ ]:
import torch
from typing import Sequence, Dict, Any

def extract_subgraph_by_ids(
    full_edge: torch.Tensor,          # shape [2, E], torch.long
    full_node: Sequence[str],         # len = N, 节点ID列表，索引与 full_edge 对应
    sub_node: Sequence[str],          # 需要的子图节点ID，顺序决定新编号
    *,
    allow_missing: bool = False       # True: 遇到不在 full_node 的ID则跳过；False: 直接报错
) -> Dict[str, Any]:
    """
    从完整图抽取子图并重编号（编号顺序 = sub_node 的顺序）。
    返回:
        {
            "edge_index": LongTensor [2, E_sub],  # 子图重编号后的边
            "node_ids": List[str],                # 子图节点ID，顺序与新编号一致
            "old_index": LongTensor [K],          # 子图节点在原图中的索引（与 node_ids 同序）
        }
    """
    if full_edge.dim() != 2 or full_edge.size(0) != 2:
        raise ValueError(f"`full_edge` must be [2, E], got {tuple(full_edge.shape)}")
    if full_edge.dtype != torch.long:
        raise ValueError(f"`full_edge` must be torch.long, got {full_edge.dtype}")

    device = full_edge.device
    N = len(full_node)

    # 1) ID -> 原图索引
    id2idx = {nid: i for i, nid in enumerate(full_node)}

    # 2) 依据 sub_node 生成去重后的原图索引列表（保持顺序 = sub_node）
    seen = set()
    keep_ids = []
    keep_idx = []
    missing = []
    for nid in sub_node:
        idx = id2idx.get(nid)
        if idx is None:
            if allow_missing:
                missing.append(nid)
                continue
            else:
                raise KeyError(f"Node id not found in full_node: {nid!r}")
        if idx not in seen:
            seen.add(idx)
            keep_ids.append(nid)
            keep_idx.append(idx)

    if len(keep_idx) == 0:
        # 空子图：返回空边，节点可为空或仅返回存在的（此处为空）
        return {
            "edge_index": torch.empty((2, 0), dtype=torch.long, device=device),
            "node_ids": [],
            "old_index": torch.empty((0,), dtype=torch.long, device=device),
        }

    old_index = torch.tensor(keep_idx, dtype=torch.long, device=device)

    # 3) 过滤边：保留两端都在子集中的边（张量方式）
    # torch.isin 需要 PyTorch>=1.10，推荐 2.x
    subset = old_index  # [K]
    mask_src = torch.isin(full_edge[0], subset)
    mask_dst = torch.isin(full_edge[1], subset)
    mask = mask_src & mask_dst
    edge_sub = full_edge[:, mask]  # [2, E_sub_old_ids]

    # 4) 重编号：构造一个大小为 N 的 remap 张量，默认 -1；子集位置写入 [0..K-1]
    remap = torch.full((N,), -1, dtype=torch.long, device=device)
    remap[subset] = torch.arange(subset.numel(), dtype=torch.long, device=device)

    if edge_sub.numel() == 0:
        edge_reindexed = torch.empty((2, 0), dtype=torch.long, device=device)
    else:
        edge_reindexed = torch.vstack((remap[edge_sub[0]], remap[edge_sub[1]]))  # [2, E_sub]

    return  edge_reindexed,  keep_ids
    

In [ ]:
sub_edge, sub_nodes = extract_subgraph_by_ids(edge_index, nodes, node_banchmark, allow_missing=True)

In [ ]:
sub_edge.shape